<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/app/prototype_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/app/prototype_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FraudLens: running our fraud detection prototype

Our final aim is a prototype that takes a transaction dataset and predicts fraud. Our experiments showed that the useful signal changes between datasets: the time gap on Sparkov, the merchant location on TabFormer. So the prototype builds both kinds of features and uses the same leak-free protocol as our notebooks.

**FraudLens** is a web app built with Streamlit. In five steps it:
1. loads a transaction table (CSV);
2. asks which column plays which role (label, time, card, amount, category, age, gender, home and merchant location);
3. checks each raw column for leakage, splits the data by time, trains a Random Forest with our settings and picks the F2 threshold on validation. The results page has a review-budget slider (how much fraud and how many fraud dollars a review budget catches), a threshold explorer on the precision-recall curve, feature importance and the train, validation and test timeline;
4. lists the riskiest transactions; clicking one opens its card, with the reasons and the card's transactions two weeks either side;
5. scores new transactions with the trained model, and lets us download the model and the scores.

Everything for the app is in the `app/` folder: the app (`fraud_app.py`), its look and charts (`ui.py`), its helper functions (`prototype.py`), this notebook and `requirements.txt`. The colours are in `.streamlit/config.toml` at the top of the repository.

**Run the cells from top to bottom.** Every cell can be run again safely: downloads and demo files that already exist are skipped.

## 1. Environment setup
We type credentials in at runtime, so they are never saved in the notebook.

In [1]:
# 1.1 Get the project repository and install the app's packages
import os
from getpass import getpass

REPO = "Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-"
REPO_DIR = f"/content/{REPO}"

if not os.path.isdir(REPO_DIR):
    # The repository is public, so a token is optional. Press Enter to skip it.
    token = getpass("GitHub token (press Enter to skip): ").strip()
    auth = f"{token}@" if token else ""
    !git clone -q https://{auth}github.com/subbareddypullalacheruvu/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

%cd {REPO_DIR}
!pip install -q -r app/requirements.txt
print("Ready:", os.getcwd())

GitHub token (press Enter to skip): ··········
/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
Ready: /content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [2]:
# 1.2 Download the demo datasets from Kaggle (skipped if already downloaded)
DATASETS = {
    "sparkov":  "kartik2112/fraud-detection",
    "tabformer": "ealtman2019/credit-card-transactions",
    "cc2026":   "mohankrishnathalla/credit-card-fraud-detection-dataset-2026",
}
missing = [k for k in DATASETS if not os.path.isdir(f"/content/data/{k}") or not os.listdir(f"/content/data/{k}")]

if missing:
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
        os.environ["KAGGLE_KEY"] = getpass("Kaggle API key: ").strip()
    for name in missing:
        print(f"Downloading {name} ...")
        !kaggle datasets download -q -d {DATASETS[name]} -p /content/data/{name} --unzip
for name in DATASETS:
    files = os.listdir(f"/content/data/{name}") if os.path.isdir(f"/content/data/{name}") else []
    print(f"{name:10s} {'ok' if files else 'MISSING'}  {files[:4]}")

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
Dataset URL: https://www.kaggle.com/datasets/ealtman2019/credit-card-transactions
License(s): other
Dataset URL: https://www.kaggle.com/datasets/mohankrishnathalla/credit-card-fraud-detection-dataset-2026
License(s): CC0-1.0
sparkov    ok  ['fraudTrain.csv', 'fraudTest.csv']
tabformer  ok  ['sd254_cards.csv', 'User0_credit_card_transactions.csv', 'credit_card_transactions-ibm_v2.csv', 'sd254_users.csv']
cc2026     ok  ['credit_card_fraud_detection_2026.csv']


## 2. Preparing the demo files
The full datasets are large, so we make smaller demo files in `app/demo_data/`. We keep every transaction of a random sample of cards (or users), so the time gaps stay correct.
- **Sparkov:** 400 cards from the training and test files together.
- **TabFormer:** 150 users from 2015 onwards. We join the users file to add the home state and the date of birth, and we combine the date and time columns into one column, so the app can read it.
- **Credit Card Fraud 2026:** the whole file (100,000 rows).

In [3]:
# 2.1 Sparkov demo file
import glob
import numpy as np
import pandas as pd

os.makedirs("app/demo_data", exist_ok=True)
path = "app/demo_data/sparkov_sample.csv"
if os.path.exists(path):
    print("Already made:", path)
else:
    rng = np.random.default_rng(42)
    sparkov = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("/content/data/sparkov/fraud*.csv"))], ignore_index=True)
    sparkov = sparkov.drop(columns=[c for c in sparkov.columns if c.startswith("Unnamed")])
    cards = rng.choice(sparkov["cc_num"].unique(), size=400, replace=False)
    sparkov_demo = sparkov[sparkov["cc_num"].isin(cards)].sort_values("trans_date_trans_time")
    sparkov_demo.to_csv(path, index=False)
    del sparkov
    print(f"Sparkov demo: {len(sparkov_demo):,} rows, {sparkov_demo['is_fraud'].sum():,} fraud")

Sparkov demo: 763,963 rows, 3,946 fraud


In [4]:
# 2.2 TabFormer demo file (reads the 24-million-row file in chunks, so it takes a few minutes)
path = "app/demo_data/tabformer_sample.csv"
if os.path.exists(path):
    print("Already made:", path)
else:
    rng = np.random.default_rng(42)
    users = pd.read_csv("/content/data/tabformer/sd254_users.csv")
    users["User"] = np.arange(len(users))
    chosen = set(rng.choice(users["User"], size=150, replace=False).tolist())

    parts = []
    for chunk in pd.read_csv("/content/data/tabformer/credit_card_transactions-ibm_v2.csv", chunksize=2_000_000):
        parts.append(chunk[chunk["User"].isin(chosen) & (chunk["Year"] >= 2015)])
    tab = pd.concat(parts, ignore_index=True)

    tab["transaction_datetime"] = (pd.to_datetime(dict(year=tab["Year"], month=tab["Month"], day=tab["Day"]))
                               + pd.to_timedelta(tab["Time"] + ":00")).astype(str)
    tab["card_id"] = tab["User"].astype(str) + "_" + tab["Card"].astype(str)
    users["dob"] = pd.to_datetime(dict(year=users["Birth Year"], month=users["Birth Month"], day=1)).astype(str)
    tab = tab.merge(users[["User", "dob", "Gender", "State"]], on="User", how="left")
    tab = tab[["transaction_datetime", "card_id", "Amount", "MCC", "Use Chip", "Merchant City", "Merchant State",
               "State", "dob", "Gender", "Is Fraud?"]].sort_values("transaction_datetime")
    tab.to_csv(path, index=False)
    print(f"TabFormer demo: {len(tab):,} rows, {(tab['Is Fraud?'] == 'Yes').sum():,} fraud")

TabFormer demo: 679,714 rows, 1,065 fraud


In [5]:
# 2.3 Credit Card Fraud 2026 demo file
path = "app/demo_data/cc2026.csv"
if os.path.exists(path):
    print("Already made:", path)
else:
    cc = pd.read_csv(sorted(glob.glob("/content/data/cc2026/*.csv"))[0])
    cc.to_csv(path, index=False)
    print(f"Credit Card Fraud 2026: {len(cc):,} rows, {cc['is_fraud'].sum():,} fraud")

Credit Card Fraud 2026: 100,000 rows, 2,055 fraud


## 3. Starting the app
Cell 3.1 starts the app and a public link in the background, checks that the app loads through the link, then prints it. Both keep running after the cell finishes, until the Colab session ends, and stopping another cell does not stop them.

The link comes from Cloudflare. It needs no account and no password, and it changes every time the app is started.

In [6]:
# 3.1 Start FraudLens and print a public link
import re
import subprocess
import time

# the address Cloudflare gives us; "api.trycloudflare.com" only appears in its error messages
LINK_PATTERN = r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com"

def http_code(url, timeout=10):
    return subprocess.run(["curl", "-s", "-o", "/dev/null", "-m", str(timeout), "-w", "%{http_code}", url],
                          capture_output=True, text=True).stdout

def app_status():
    return http_code("http://127.0.0.1:8501/_stcore/health", 5) == "200"

def stop_app():
    # list arguments, no shell: pkill never matches its own command line
    subprocess.run(["pkill", "-f", "streamlit run app/fraud_app.py"])
    subprocess.run(["pkill", "-f", "cloudflared tunnel"])
    time.sleep(2)

def start_link():
    # start a Cloudflare link and return it once a page loads through it
    if not os.path.exists("cloudflared"):
        subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
                       "cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True)
    subprocess.run(["pkill", "-f", "cloudflared tunnel"])
    subprocess.Popen("./cloudflared tunnel --no-autoupdate --url http://127.0.0.1:8501 > cloudflared.log 2>&1",
                     shell=True, start_new_session=True)
    url = None
    for _ in range(60):
        time.sleep(1)
        log = open("cloudflared.log").read()
        if "failed to request quick Tunnel" in log:
            print("Cloudflare could not make a link this time.")
            return None
        found = re.search(LINK_PATTERN, log)
        if found and "Registered tunnel connection" in log:
            url = found.group(0)
            break
    if not url:
        print("Cloudflare did not connect in time.")
        return None
    print("Link made, checking that the app loads through it ...")
    for _ in range(30):
        if http_code(url + "/_stcore/health") == "200":
            return url
        time.sleep(2)
    print("The link exists but does not reach the app yet:", url)
    return None

def start_app():
    stop_app()
    # The app listens on all connections, so both the Cloudflare link and Colab's own view
    # (cell 3.2) can reach it. CORS and XSRF checks are off for Colab's view.
    subprocess.Popen(
        "streamlit run app/fraud_app.py --server.port 8501 "
        "--server.headless true --server.maxUploadSize 1000 "
        "--server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1",
        shell=True, start_new_session=True)
    for _ in range(60):
        time.sleep(1)
        if app_status():
            break
    if not app_status():
        print("The app did not start. Its log:")
        print(open("streamlit.log").read()[-2000:])
        return None
    print("The app is running.")
    return start_link()

APP_URL = start_app()
if APP_URL:
    print()
    print("Open FraudLens at:", APP_URL)
    print("The link stays live until the Colab session ends.")
else:
    print("Use cell 3.2 to open the app inside this notebook, or run cell 3.3 to try the link again.")

The app is running.
Link made, checking that the app loads through it ...

Open FraudLens at: https://league-exhibits-generated-eventually.trycloudflare.com
The link stays live until the Colab session ends.


In [7]:
# 3.2 Optional: show the app inside this notebook instead of a separate tab
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=900)

<IPython.core.display.Javascript object>

In [8]:
# 3.3 Check the app, or restart it if the link stops working
if not app_status():
    print("The app had stopped. Restarting it ...")
    APP_URL = start_app()
else:
    print("The app is running.")
    found = re.findall(LINK_PATTERN, open("cloudflared.log").read()) if os.path.exists("cloudflared.log") else []
    if found and http_code(found[-1] + "/_stcore/health") == "200":
        APP_URL = found[-1]
    else:
        print("The link is not working, so making a new one ...")
        APP_URL = start_link()
print("Open FraudLens at:", APP_URL) if APP_URL else print("No working link yet. Use cell 3.2, or run this cell again in a minute.")

The app is running.
Open FraudLens at: https://league-exhibits-generated-eventually.trycloudflare.com


## 4. Using the app with the demo files
1. **Load data:** choose a demo file under *Or use a demo file* and click **Load demo file**.
2. **Match columns:** most columns are filled in automatically. Check them against the table below.
3. **Train and evaluate:** click **Train the model**. On the Sparkov demo this takes under a minute. Then try the review-budget and threshold sliders.
4. **Review alerts:** click any row to open that card.
5. **Score new data:** upload a CSV with the same columns to score it with the trained model.

| Demo file | Label | Time | Card | Amount | Category | Customer | Location |
|---|---|---|---|---|---|---|---|
| `sparkov_sample.csv` | `is_fraud` | `trans_date_trans_time` | `cc_num` | `amt` | `category` | `dob`, `gender` | `lat`, `long`, `merch_lat`, `merch_long` |
| `tabformer_sample.csv` | `Is Fraud?` | `transaction_datetime` | `card_id` | `Amount` | `MCC` | `dob`, `Gender` | home state `State`, merchant state `Merchant State` |
| `cc2026.csv` | `is_fraud` | `transaction_datetime` | `card_id` | `transaction_amount` | `merchant_category` | `customer_age`, `customer_gender` | merchant state or country `merchant_country` (home state left as *(none)*) |

Sparkov has merchant coordinates but no merchant state, so the app builds the distance from home instead of the state features. Credit Card Fraud 2026 gives the merchant country, so the app builds only the abroad feature there.

**Each run of cell 3.1 makes a new link** and closes the old one, so always open the most recent link.

**If the page looks broken or half-loaded,** press Ctrl + Shift + R (Cmd + Shift + R on a Mac) to reload it fully. **If the link stops working,** run cell 3.3.

In [10]:
# 4.1 Stop the app and the public link when you are done
# "Run all" does not stop the app. To stop it, change False to True and run this cell.
STOP_APP = True

if STOP_APP:
    stop_app()
    print("Stopped.")
else:
    print("FraudLens is still running. To stop it, set STOP_APP = True and run this cell.")

Stopped.
